# 👑 Orbit Wars: OMEGA v5 (Supreme Domination Engine)
**Kaggle Public Score: 644.7**

OMEGA v5 is a highly aggressive, hyper-optimized evolution of the base strategic agent. It introduces features like **Endgame State Awareness**, **Speed Bonus Mechanics**, and **Dynamic Defense Scaling** to dominate the battlefield.

## 1. Dynamic Defense Scaling
Unlike passive agents that keep a static number of ships for defense on every planet, OMEGA recognizes its position. If a planet is far behind the frontlines (`dist > 40`), it releases its garrison to join the action. Frontline planets multiply their buffers to absorb incoming shockwaves.

In [ ]:
# --- EXCERPT: DYNAMIC DEFENSE SCALING ---
# Dynamic Defense: Rearline planets shouldn't hoard ships, funnel them to the action
dist_to_closest_enemy = min([get_dist(src.x, src.y, e.x, e.y) for e in enemy_planets] + [999.0])
buffer_multiplier = 1.0 if dist_to_closest_enemy > 40 else 3.0

reserve = src.production * buffer_multiplier + max(0, src.incoming_enemy - src.incoming_allied)
available = max(0, src.ships - max(3, reserve))

## 2. Distance Penalty & Endgame State Awareness
The ROI calculation dynamically shifts based on the game state. In the early/mid-game, OMEGA slightly penalizes long-distance travel (`eta ** 1.2`) to maintain tight, localized control. However, when the game enters the final 60 steps (`Endgame`), production ceases to matter, and OMEGA switches to pure ship-hunting mode to maximize final score.

In [ ]:
# --- EXCERPT: ENDGAME AND DISTANCE SCORING ---
time_left = TOTAL_STEPS - (step + eta)

# Endgame Mode
if time_left_global < 60:
    score = (tgt.ships * 15) / (needed * (eta + 1))
else:
    # Slightly penalize long distances to prevent over-stretching (eta ** 1.2)
    score = (tgt.production * 15 * time_left) / (needed * (eta ** 1.2) + 1)
    
    if tgt.owner == -1: 
        score *= 1.5 # High incentive for neutral planets
    elif tgt.id in comet_ids: 
        score *= 0.6 # Comets are risky as they disappear

## 3. Fleet Acceleration Bonus (Over-provisioning)
In "Orbit Wars", fleet speed scales logarithmically with the number of ships. OMEGA exploits this mechanic. If it has a vast surplus of ships, it intentionally sends **10% more ships than mathematically needed** (`amt * 1.1`). This "over-provisioning" increases the fleet's velocity, allowing it to hit the target faster than the enemy expects.

In [ ]:
# --- EXCERPT: FLEET ACCELERATION ---
# Speed Bonus: If we have a massive surplus, send 10% extra ships to make the fleet fly faster
send_amt = amt
if available > amt * 1.5 and available > 30:
    send_amt = min(available, int(amt * 1.1))
    
actions.append([int(src.id), float(angle), int(send_amt)])
src.ships -= send_amt
p_map[tid].incoming_allied += send_amt # Protects against overkill